# Pandas — analiză de date explicată pas cu pas

**ONIA · curs aplicativ, 14 secțiuni.** Notebook didactic complementar pentru lecția HTML „Pandas — ghid practic pentru DataFrame”. Include exemple numerice, verificări și reprezentări vizuale.

Materialul de bază din proiect este notebookul **„Pandas – A Quick Introduction”, de Dan Novischi**. Acest notebook este o **extindere pedagogică nouă**, nu o suprascriere a fișierului original. Date demonstrative mici sunt utilizate pentru a vedea exact fiecare rezultat; apoi lucrăm cu `avocado.csv` din proiect.

Necesită: `pandas`, `numpy`, `matplotlib`. Se rulează cu Python 3 și Jupyter. Pentru instalare, în terminal: `python -m pip install pandas numpy matplotlib notebook`.

## 0. Pregătirea mediului și localizarea fișierului

Notebookul este stocat în `notebooks/`, iar setul de date în `datasets/` din rădăcina proiectului. În funcție de directorul de lucru Jupyter, calea va fi `../datasets/avocado.csv` sau `datasets/avocado.csv`. Verificăm ambele, astfel încât exemplul să funcționeze în proiect.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidati = [Path("../datasets/avocado.csv"), Path("datasets/avocado.csv")]
csv_avocado = next((p for p in candidati if p.is_file()), None)
if csv_avocado is None:
    raise FileNotFoundError("Nu găsesc datasets/avocado.csv. Rulează notebookul în proiectul ONIA.")
avocado = pd.read_csv(csv_avocado, index_col=0)
print("Pandas", pd.__version__, "| forma", avocado.shape)
assert avocado.shape == (18249, 13)
assert avocado.index.is_unique is False

## 1. `Series` și `DataFrame`

O `Series` păstrează valori și etichete de index. Un `DataFrame` păstrează mai multe coloane, fiecare cu propriul tip. `shape` întoarce `(număr_rânduri, număr_coloane)`; `iloc` folosește **poziții de la 0**.

Pentru `s = [1.2, 1.4, 1.1]`, media aritmetică este `(1.2+1.4+1.1)/3 = 3.7/3 ≈ 1.2333`. Nu confunda media cu suma ori mediana.

In [ ]:
s = pd.Series([1.2, 1.4, 1.1], name="pret")
df_orase = pd.DataFrame({"oras": ["București", "Cluj", "Iași"],
                        "scor": [0.91, 0.88, 0.86]})
print("Series:")
print(s)
print("Media =", s.mean())
print("DataFrame:")
print(df_orase)
print("shape =", df_orase.shape, "| index =", df_orase.index.tolist())
print("Series vs DataFrame:", type(df_orase["scor"]).__name__,
      type(df_orase[["scor"]]).__name__)
assert df_orase["scor"].shape == (3,)
assert df_orase[["scor"]].shape == (3, 1)

## 2. Citirea și exportul unui fișier CSV

Într-un CSV, liniile sunt observații, iar câmpurile sunt separate de obicei prin virgulă. `pd.read_csv()` construiește un DataFrame. `index_col=0` folosește prima coloană din fișier drept etichetă a rândurilor (nu garantează că etichetele sunt unice). `to_csv(index=False)` exportă fără această coloană de index.

In [ ]:
print("Rânduri, coloane:", avocado.shape)
print("Primele coloane:", avocado.columns[:5].tolist())
print(avocado[["Date", "AveragePrice", "region"]].head(3))
# În loc de a scrie pe disc la rularea tuturor celulelor, generăm CSV în memorie:
print("\
Export demonstrativ fără index:")
print(avocado.head(2).to_csv(index=False)[:300])

## 3. Inspecția tabelului: `head`, `tail`, `info`, `describe`

`head(n)` returnează primele n rânduri **în ordinea curentă**, `tail(n)` ultimele n. `info()` indică tipurile și valorile nenule. `describe()` calculează pentru coloane numerice `count`, `mean`, `std`, `min`, percentilele 25%, 50%, 75% și `max`.

Exemplu controlat: `[1, 2, 3, 4, 10]` are media `4`, mediana `3`, extrema maximă `10` și abaterea standard a eșantionului `sqrt(50/4) ≈ 3.5355`.

In [ ]:
print("primele 2 rânduri:\n", avocado.head(2))
print("ultimele 2 rânduri:\n", avocado.tail(2))
print("Sumar din info():")
avocado.info()
print("Descrierea coloanei AveragePrice:")
print(avocado["AveragePrice"].describe())
ex_stat = pd.Series([1, 2, 3, 4, 10])
print("Control manual:", ex_stat.agg(["count", "mean", "median", "std", "min", "max"]))
assert np.isclose(ex_stat.std(), np.sqrt(12.5))

## 4. Selecție cu `loc` și `iloc`

`loc` selectează după **etichete**, `iloc` după **poziții**. Pentru `iloc[a:b]`, limita `b` este exclusă. Pentru `loc[eticheta_a:eticheta_b]`, capătul este inclus în situațiile uzuale de index ordonat cu etichete existente. Când indexul are duplicate, `loc` poate returna **mai multe rânduri**.

În **fișierul avocado real**, prima coloană conține etichete numerice repetate (de exemplu 0). Prin urmare, `avocado.loc[0, "AveragePrice"]` este o serie, nu un număr unic. Pentru primul rând se folosește `iloc[0]`.

In [ ]:
elevi = pd.DataFrame({"nume": ["Ana", "Bogdan", "Cora"],
                      "scor": [7, 8, 9], "proiect": [9, 10, 8]},
                     index=["E01", "E02", "E03"])
print(elevi)
print("loc E02/scor:", elevi.loc["E02", "scor"])
print("iloc 1,1:", elevi.iloc[1, 1])
print("iloc 0:2, 1:3:\n", elevi.iloc[0:2, 1:3])
print("loc E01:E02, scor:proiect:\n", elevi.loc["E01":"E02", "scor":"proiect"])
print("Avocado: index unic?", avocado.index.is_unique)
print("Poziția primului preț:", avocado.iloc[0]["AveragePrice"])
print("Câte rânduri au eticheta 0?", len(avocado.loc[0]))
assert len(avocado.loc[0]) > 1

## 5. Filtrarea — masca booleană, pas cu pas

O expresie precum `elevi["scor"] >= 8` se aplică fiecărei valori și creează o `Series` de `True`/`False`. `sum()` pe o mască numără valorile `True` (1) și ignoră `False` (0). Indexarea cu masca păstrează doar rândurile unde valoarea este `True`.

Pentru două condiții folosim `&` (și), `|` (sau), `~` (negație), fiecare comparație între paranteze. În Pandas, `and`/`or` nu combină element-cu-element serii booleene.

In [ ]:
elevi = pd.DataFrame({"nume": ["Ana", "Bogdan", "Cora", "Daria"],
                      "scor": [7.5, 8.0, 9.5, 6.0],
                      "proiect": [9, 7, 10, 10]})
masca = elevi["scor"] >= 8
print("Tabel inițial:\n", elevi)
print("Mască scor >= 8:\n", masca)
print("Număr elevi cu scor >= 8:", masca.sum())
print("Rânduri selectate:\n", elevi[masca])
print("Scor >= 8 ȘI proiect >= 8:\n",
      elevi[(elevi["scor"] >= 8) & (elevi["proiect"] >= 8)])
assert masca.tolist() == [False, True, True, False]

## 6. Index și date calendaristice

O dată citită din CSV poate fi text (`object`); `pd.to_datetime()` o convertește la un tip temporal. `.dt.year` și `.dt.month` extrag componente ale unei coloane temporale, `set_index("Date")` mută acea coloană în index, iar `reset_index()` o readuce în tabel.

Când conversia eșuează, `errors="coerce"` poate înlocui data invalidă cu `NaT`. `set_index()` nu sortează automat tabelul.

In [ ]:
date = pd.DataFrame({"Date": ["2026-01-07", "2026-01-14", "2026-01-21"],
                     "pret": [2.0, 2.5, 3.0]})
print("Înainte:\n", date.dtypes)
date["Date"] = pd.to_datetime(date["Date"], errors="coerce")
date["an"] = date["Date"].dt.year
date["luna"] = date["Date"].dt.month
print("După conversie:\n", date)
pe_date = date.set_index("Date")
print("Cu index temporal:\n", pe_date)
print("Revenire la coloană:\n", pe_date.reset_index())

## 7. Sortare și coloane calculate

`sort_values("pret", ascending=False)` întoarce un tabel ordonat descrescător după preț, fără modificarea implicită a originalului. O atribuire precum `df["volum_mii"] = df["volum"] / 1000` este vectorizată — fiecare rând este transformat conform aceleiași reguli.

In [ ]:
produse = pd.DataFrame({"produs": ["A", "B", "C"],
                         "pret": [4.0, 2.0, 5.0],
                         "volum": [1200, 800, 1500]},
                        index=["E01", "E02", "E03"])
print("Original:\n", produse)
print("Sortare descrescătoare:\n", produse.sort_values("pret", ascending=False))
produse["volum_mii"] = produse["volum"] / 1000
print("Coloană derivată:\n", produse)
assert produse.loc["E01", "volum_mii"] == 1.2

## 8. Agregări și sume cumulative

`mean()` calculează media, `median()` valoarea centrală, `count()` numără valorile **nelipsă**, `nunique()` valorile distincte. `cumsum()` calculează câte un total parțial la fiecare poziție.

Pentru `[2, 5, 1, 4]` obținem: `[2, 2+5, 2+5+1, 2+5+1+4] = [2, 7, 8, 12]`. Ordinea rândurilor este importantă pentru o sumă cumulativă.

In [ ]:
numere = pd.Series([2, 5, 1, 4])
print("sum:", numere.sum(), "mean:", numere.mean(),
      "median:", numere.median(), "count:", numere.count())
print("cumsum:\n", numere.cumsum())
assert numere.cumsum().tolist() == [2, 7, 8, 12]
plt.figure(figsize=(7, 3))
plt.bar(range(len(numere)), numere.cumsum(), edgecolor="black")
plt.xticks(range(len(numere)))
plt.ylabel("Suma până la poziția curentă")
plt.xlabel("Poziția")
plt.title("Cum crește suma cumulativă")
plt.tight_layout()
plt.show()

## 9. Grupări: `groupby()` = Split → Apply → Combine

`groupby("regiune")["pret"].mean()` împarte tabelul după valoarea coloanei `regiune`, calculează media pentru fiecare grup, apoi reunește rezultatele. Dacă grupa A are 4, 2, 6, atunci media este `(4+2+6)/3=4`; pentru grupa B cu 3, 5 media este `(3+5)/2=4`. `agg()` permite mai multe statistici deodată. Atenție: media mediilor nu este mereu media globală atunci când grupele au dimensiuni diferite.

In [ ]:
vanzari = pd.DataFrame({"regiune": ["A", "A", "B", "B", "A"],
                        "tip": ["organic", "conventional", "organic", "conventional", "organic"],
                        "pret": [4, 2, 3, 5, 6]})
print(vanzari)
print("Media per regiune:\n", vanzari.groupby("regiune")["pret"].mean())
print("Suma per regiune:\n", vanzari.groupby("regiune")["pret"].sum())
print("Statistici:\n", vanzari.groupby("regiune")["pret"].agg(["count", "mean", "median", "max"]))
assert vanzari.groupby("regiune")["pret"].sum().to_dict() == {"A": 12, "B": 8}

## 10. Grafice: ce întrebare răspunde fiecare?

- **Bară**: comparația volumelor totale pe categorii;
- **Linie**: evoluția unui indicator după dată;
- **Histogramă**: frecvențe în intervale numerice (distribuție);
- **Scatter**: legătura între două variabile numerice (puncte).

Folosește axele etichetate, scări adecvate și, la nevoie, agregarea înainte de reprezentare. Un grafic scatter nu demonstrează singur cauzalitate.

In [ ]:
# Demo pe date mici pentru a evita încărcarea graficelor cu mii de observații.
fig, ax = plt.subplots(figsize=(6, 3))
vanzari.groupby("regiune")["pret"].sum().plot(kind="bar", ax=ax)
ax.set_title("Totalul prețurilor pe regiuni — exemplu didactic")
ax.set_xlabel("Regiune")
ax.set_ylabel("Sumă prețuri (unități demonstrative)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
numere.plot(kind="hist", bins=3, ax=ax, edgecolor="black")
ax.set_title("Histogramă pentru [2, 5, 1, 4]")
ax.set_xlabel("Valoare")
ax.set_ylabel("Frecvență")
plt.tight_layout()
plt.show()

## 11. Medii mobile: `rolling()`

O medie mobilă cu fereastra `w=3` folosește, la fiecare rând, ultimele 3 valori. Pentru `[2,4,6,8,10]` avem: primele două rezultate `NaN` (nu sunt 3 valori), apoi `(2+4+6)/3=4`, `(4+6+8)/3=6`, `(6+8+10)/3=8`.

`rolling(window=14)` se referă la **14 observații**, nu automat la 14 zile. Înainte de folosire într-o serie temporală se sortează datele; se păstrează, de regulă, separate regiunile sau alte categorii distincte.

In [ ]:
preturi = pd.Series([2, 4, 6, 8, 10], name="pret")
print("rolling(3):\n", preturi.rolling(window=3).mean())
print("rolling(3, min_periods=1):\n", preturi.rolling(window=3, min_periods=1).mean())
assert preturi.rolling(3).mean().iloc[2:].tolist() == [4., 6., 8.]
fig, ax = plt.subplots(figsize=(7, 3))
preturi.plot(marker="o", ax=ax, label="Valoarea")
preturi.rolling(window=3).mean().plot(marker="s", ax=ax, label="Media mobilă (3)")
ax.set_xlabel("Poziția observației")
ax.legend()
plt.tight_layout()
plt.show()

## 12. Valori lipsă: `isna`, `dropna`, `fillna`

`NaN` este informație lipsă, nu o valoare 0. `isna()` returnează o mască booleană; `isna().sum()` numără valorile lipsă din fiecare coloană. `dropna()` elimină implicit rânduri cu cel puțin o valoare lipsă. `fillna()` completează după o regulă explicită.

Exemplu: notele `[8, NaN, 10]`. Media calculată pe observații cunoscute este `(8+10)/2=9`. Completarea cu 9 este o **estimare**, nu o măsurătoare reală.

In [ ]:
note = pd.DataFrame({"elev": ["Ana", "Bogdan", "Cora"],
                     "nota": [8.0, np.nan, 10.0]})
print("Inițial:\n", note)
print("Mască de lipsuri:\n", note.isna())
print("Câte lipsesc?\n", note.isna().sum())
print("După eliminarea rândurilor incomplete:\n", note.dropna())
medie_cunoscute = note["nota"].mean()
print("Media valorilor cunoscute:", medie_cunoscute)
print("Completare cu media:\n", note.fillna({"nota": medie_cunoscute}))
print("Date reale avocado: valori lipsă totale =", int(avocado.isna().sum().sum()))
assert avocado.isna().sum().sum() == 0

## 13. One-hot encoding pentru categorii

`pd.get_dummies()` transformă o coloană categorială în indicatori. Pentru tipurile `organic` și `conventional`, obținem două coloane cu 0 și 1. `dtype=int` solicită valori întregi pentru indicatori. Aceasta evită presupunerea unei ordini între categorii pur nominale.

La folosirea în machine learning trebuie aliniate coloanele la datele de test și evitată prelucrarea care folosește informații din setul de test la ajustarea transformărilor.

In [ ]:
etichete = pd.DataFrame({"tip": ["organic", "conventional", "organic"]})
encodat = pd.get_dummies(etichete, columns=["tip"], dtype=int)
print("Inițial:\n", etichete)
print("One-hot:\n", encodat)
assert sorted(encodat.columns) == ["tip_conventional", "tip_organic"]
assert encodat["tip_organic"].tolist() == [1, 0, 1]

## 14. Exercițiu rezolvat pe fișierul avocado

**Cerinta 1:** determină regiunea cu cel mai mare volum **total**. Alege `groupby('region')['Total Volume'].sum()` și abia apoi `idxmax()`.

**Cerinta 2:** calculează prețul mediu pe fiecare `year` și `type`. `as_index=False` păstrează grupele în coloane pentru un export facil.

**Cerinta 3:** verifică valorile lipsă înainte de preprocesare. Fișierul furnizat în proiect are zero valori lipsă după `read_csv`.

**Cerinta 4:** exportă un rezumat în memorie fără index pentru a putea verifica primele linii.

In [ ]:
total_regiuni = avocado.groupby("region")["Total Volume"].sum()
regiune_maxima = total_regiuni.idxmax()
print("Regiunea cu volum maxim:", regiune_maxima)
print("Volumul maxim total:", total_regiuni.max())

pret_mediu = avocado.groupby(["year", "type"], as_index=False)["AveragePrice"].mean()
print("Prețuri medii pe an și tip:\n", pret_mediu)
print("Lipsuri pe coloane:\n", avocado.isna().sum())
print("Primele linii de CSV exportat:\n", pret_mediu.to_csv(index=False)[:350])

## 15. Provocări pentru lucru independent

1. Filtrează numai observațiile din `Atlanta` și tipul `organic`; calculează prețul lor mediu.
2. Află care an are cel mai mare volum total și justifică de ce trebuie calculată suma pe an.
3. Transformă `Date` într-un tip temporal și construiește o coloană `month`.
4. Pentru o singură regiune și un singur tip, sortează după dată și calculează media mobilă a prețului pe 4 observații.
5. Verifică printr-un exemplu când media mediilor grupurilor diferă de media tuturor valorilor.

**De reținut:** Pandas păstrează etichetele de index; `loc` și `iloc` nu sunt interschimbabile, iar `groupby()` trebuie aplicat înainte de a interpreta totaluri pe categorii.

In [ ]:
# Verificări recapitulative, fără afișarea soluțiilor la provocări
assert not avocado.empty
assert avocado["AveragePrice"].dtype.kind in "fi"
assert avocado["Total Volume"].ge(0).all()
assert avocado.index.is_unique is False
print("Verificările de bază au trecut. Poți rezolva exercițiile 1–5 într-o celulă nouă.")